In [ ]:
import numpy as np
from scipy import signal
from statsmodels.tsa.arima_process import ArmaProcess

from data import get_switzerland_temperature, get_victoria_electricity_demand, get_periodic_process
from utils import get_figure, time_plot, arma_theoretical_spectral_density, plot_fft, moving_average_smoothing

## Spectral leakage

In [ ]:
n = 100
x_t, x_name = get_periodic_process(seed=10, npoints=n, sd=1, period=12)

fig, (ax1, ax2, ax3) = get_figure(ncols=3, figsize=(18, 4))
time_plot(
    np.arange(len(x_t)),
    x_t,
    title=f"$x_t$, {x_name}",
    prefix_title=True,
    ax=ax1,
)
plot_fft(
    ax2,
    x_t,
    periodogram=False,
)
plot_fft(
    ax3,
    x_t,
    periodogram=False,
    log_scale=True,
)
fig.tight_layout()

In [ ]:
omega = np.linspace(-.5, .5, 1000)
dirichlet_kernel = np.sin(np.pi * omega * n) / (n * np.sin(np.pi * omega))

time_plot(
    omega,
    dirichlet_kernel,
    title="Rescaled Dirichlet Kernel",
    xlabel=r"Frequency $\omega$ [cycles per unit time]",
    ylabel=r"$D_n(\omega)$",
)

## Periodogram bias

In [ ]:
frequencies = np.linspace(-.45, .45, 1000)
fig, axs = get_figure(nrows=2, ncols=3, figsize=(18, 8))
for i, n in enumerate([10, 50, 100]):
    fejer_kernel = np.sin(np.pi * frequencies * n)**2 / (n * np.sin(np.pi * frequencies)**2)
    time_plot(
        frequencies,
        fejer_kernel,
        title=f"Féjer Kernel with $n={n}$",
        xlabel=r"Frequency $\omega$ [cycles per unit time]",
        ylabel=r"$nD_n^2(\omega)$",
        ax=axs[i],
    )
    time_plot(
        frequencies,
        fejer_kernel,
        title=f"Féjer Kernel with $n={n}$",
        xlabel=r"Frequency $\omega$ [cycles per unit time]",
        ylabel=r"$log(nD_n^2(\omega))$",
        ax=axs[i+3],
    )
    axs[i+3].set_yscale('log')
fig.tight_layout()

## Tapering

In [ ]:
np.random.seed(42)
n = 1000
sigma = 1
arma_configs = [
    {"phi": [0.5, -0.4], "theta": [0.3], "label": "ARMA(2,1)"},
    {"phi": [0.7], "theta": [0.2, -0.1], "label": "ARMA(1,2)"},
    {"phi": [0.6, -0.5, 0.1], "theta": [], "label": "ARMA(3,0)"},
    {"phi": [], "theta": [0.4, -0.3], "label": "ARMA(0,2)"},
]
frequencies = np.linspace(0, 0.5, n)
nrows= np.ceil(len(arma_configs)/2)
fig, axs = get_figure(nrows=len(arma_configs)//2, ncols=2, figsize=(12, 8))
for ax, config in zip(axs, arma_configs):
    phi = config["phi"]
    theta = config["theta"]
    label = config["label"]
    
    model = ArmaProcess(
        ar=np.r_[1, -np.array(phi)],
        ma=np.r_[1, np.array(theta)],
    )
    arma_series = model.generate_sample(nsample=n)
    spectral_density = arma_theoretical_spectral_density(frequencies, sigma, phi, theta)
    # Multiply by two for the negative frequencies, divide by n to relate to the periodogram since P(w) = f(w)/n
    spectral_density = 2*spectral_density/n
    
    tapered_series = signal.windows.hann(n) * arma_series
    
    plot_fft(
        ax,
        tapered_series,
        periodogram=True,
    )
    time_plot(
        frequencies,
        spectral_density,
        title=f"Spectral Density of Tapered {label} Process",
        xlabel="Frequency",
        ylabel="Spectral Density",
        label="Theoretical",
        ax=ax,
    )
fig.tight_layout()

In [ ]:
def compare_with_taper(data, title, ylabel, smoothing=False, **kwargs):
    data = data - data.mean()
    tapered_data = data * signal.windows.hann(len(data))

    fig, axs = get_figure(nrows=2, ncols=3, figsize=(18, 8))
    time_plot(
        x=data.index,
        y=data,
        title=title,
        xlabel='Time',
        ylabel=ylabel,
        ax=axs[0],
    )
    plot_fft(
        axs[1],
        data,
        sample_spacing=1 / 12,
        sample_spacing_name="year",
        periodogram=True,
        show_estimated_density=smoothing,
        **kwargs
    )
    plot_fft(
        axs[2],
        data,
        sample_spacing=1 / 12,
        sample_spacing_name="year",
        periodogram=True,
        log_scale=True,
        show_estimated_density=smoothing,
        **kwargs
    )
    time_plot(
        x=tapered_data.index,
        y=tapered_data,
        title=f"Tapered {title}",
        xlabel='Time',
        ylabel=ylabel,
        ax=axs[3],
    )
    plot_fft(
        axs[4],
        tapered_data,
        sample_spacing=1 / 12,
        sample_spacing_name="year",
        periodogram=True,
        show_estimated_density=smoothing,
        **kwargs
    )
    plot_fft(
        axs[5],
        tapered_data,
        sample_spacing=1 / 12,
        sample_spacing_name="year",
        periodogram=True,
        log_scale=True,
        show_estimated_density=smoothing,
        **kwargs
    )
    if smoothing:
        axs[1].legend()
    fig.tight_layout()

In [ ]:
data = get_switzerland_temperature().set_index('dt').asfreq('ME')['AverageTemperature']
compare_with_taper(data, 'Zero-mean Monthly average temperature in Switzerland', 'Average Temperature (°C)')

In [ ]:
data = get_victoria_electricity_demand()["OperationalLessIndustrial"]
compare_with_taper(data, 'Zero-mean Victoria monthly electricity demand', 'Demand (MW)')

## Smoothed periodogram

In [ ]:
np.random.seed(42)
n=1000
sigma = 1
frequencies = np.linspace(0, 0.5, n)
config = {"phi": [0.5, -0.4], "theta": [0.3], "label": "ARMA(2,1)"}
phi = config["phi"]
theta = config["theta"]
label = config["label"]

model = ArmaProcess(
    ar=np.r_[1, -np.array(phi)],
    ma=np.r_[1, np.array(theta)],
)
arma_series = model.generate_sample(nsample=n)
spectral_density = arma_theoretical_spectral_density(frequencies, sigma, phi, theta)
# Multiply by two for the negative frequencies, divide by n to relate to the periodogram since P(w) = f(w)/n
spectral_density = 2*spectral_density/n
fig, axs = get_figure(nrows=2, ncols=2, figsize=(12, 8))
for ax, wsize in zip(axs, [16, 32, 64, 128]):
    xf, _, periodogram = plot_fft(
        ax,
        arma_series,
        periodogram=True,
        return_fft=True,
    )
    time_plot(
        frequencies,
        spectral_density,
        title=f"Smoothed periodogram with {wsize}-MA ($L={wsize}$)",
        xlabel="Frequency",
        ylabel="Spectral Density",
        label="Theoretical",
        ax=ax,
    )
    
    _, _, smoothed_periodogram = moving_average_smoothing(periodogram, window_size=wsize, mode='same')
    ax.plot(xf, smoothed_periodogram, label="Smoothed periodogram", color='black')
    ax.legend()
fig.tight_layout()

In [ ]:
def compare_with_smoothing(data, title, ylabel, smoothing_window_percent):
    data = data - data.mean()
    data = data * signal.windows.hann(len(data))

    fig, axs = get_figure(nrows=2, ncols=3, figsize=(18, 8))
    time_plot(
        x=data.index,
        y=data,
        title=title,
        xlabel='Time',
        ylabel=ylabel,
        ax=axs[0],
    )
    plot_fft(
        axs[1],
        data,
        sample_spacing=1 / 12,
        sample_spacing_name="year",
        periodogram=True,
        show_estimated_density=True,
        smoothing_iteration=1,
        smoothing_window_percent=smoothing_window_percent,
    )
    plot_fft(
        axs[2],
        data,
        sample_spacing=1 / 12,
        sample_spacing_name="year",
        periodogram=True,
        log_scale=True,
        show_estimated_density=True,
        smoothing_iteration=1,
        smoothing_window_percent=smoothing_window_percent,
    )
    time_plot(
        x=data.index,
        y=data,
        title=f"Tapered {title}",
        xlabel='Time',
        ylabel=ylabel,
        ax=axs[3],
    )
    plot_fft(
        axs[4],
        data,
        sample_spacing=1 / 12,
        sample_spacing_name="year",
        periodogram=True,
        show_estimated_density=True,
        smoothing_iteration=2,
        smoothing_window_percent=smoothing_window_percent,
    )
    plot_fft(
        axs[5],
        data,
        sample_spacing=1 / 12,
        sample_spacing_name="year",
        periodogram=True,
        log_scale=True,
        show_estimated_density=True,
        smoothing_iteration=2,
        smoothing_window_percent=smoothing_window_percent,
    )
    axs[1].legend()
    axs[4].legend()
    fig.tight_layout()

In [ ]:
data = get_victoria_electricity_demand()["OperationalLessIndustrial"]
compare_with_smoothing(data, 'Zero-mean Victoria monthly electricity demand', 'Demand (MW)', 0.04)

## Linear filters

In [ ]:
frequencies = np.linspace(-0.5, 0.5, 1000)
A_squared = 2*(1-np.cos(2*np.pi*frequencies))
time_plot(
    frequencies,
    A_squared,
    title="Squared frequency response of the first order differencing filter",
    xlabel="Frequency",
    ylabel="Frequency response",
)

In [ ]:
frequencies = np.linspace(-0.5, 0.5, 1000)
A = (1+np.cos(12*np.pi*frequencies)+2*sum(np.cos(2*np.pi*frequencies*k) for k in range(1,6)))/12
A_squared = A**2
time_plot(
    frequencies,
    A_squared,
    title="Squared frequency response of the modified Daniell kernel with $m=6$ filter",
    xlabel="Frequency",
    ylabel="Frequency response",
)